# GenImage — Complete Dataset & Balanced Experiments (third Google account)

**Run this notebook after Midjourney has finished extracting.** It reproduces the workflow shown in the project screenshots: verify all 8 generators, explore image counts and file types, create balanced experiment manifests, verify those splits, and load a sample batch into PyTorch.

**This notebook is self-contained:** it does **not** require `genimage_pipeline.zip`, `prepare_genimage.py`, or `dataset.py`. All the code is included below.

**Access needed:** the third account must have access to the original `GenImage_Compact` Drive folder and add a shortcut in **My Drive** called `GenImage_Compact_shared`. **Editor access to the compact folder is required because experiment manifests will be saved there.**

**Important:** experiment CSVs are stored in a **new `_experiment_manifests` subfolder of your original shared `GenImage_Compact`**. Images stay in their existing folders; no dataset copy is created. The third account needs **Editor** access to save these CSVs.


## 1 — Mount Google Drive and configure the dataset
Run this cell first. If Colab prompts you to connect Google Drive, authorize the third person’s account. Do not upload or copy the dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from collections import Counter
import hashlib, json, random, shutil, time
import pandas as pd
from IPython.display import display

GENERATORS = [
    'ADM', 'BigGAN', 'glide', 'Midjourney',
    'stable_diffusion_v_1_4', 'stable_diffusion_v_1_5',
    'VQDM', 'wukong'
]
TARGETS = {
    ('train', 'ai'): 1500,
    ('train', 'nature'): 1500,
    ('val', 'ai'): 500,
    ('val', 'nature'): 500,
}
EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

# Prefer the shortcut to the owner's ORIGINAL compact dataset.
ROOT_CANDIDATES = [
    Path('/content/drive/MyDrive/GenImage_Compact_shared'),
    Path('/content/drive/MyDrive/GenImage_Compact'),
]
DATA_ROOT = next((p for p in ROOT_CANDIDATES if (p / 'ADM' / 'train' / 'ai').is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        'Dataset shortcut not found. Add the shared GenImage_Compact folder '
        'to My Drive as GenImage_Compact_shared, then remount Drive.'
    )
print('Reading ORIGINAL shared dataset from:', DATA_ROOT)
print('This notebook does not copy the images.')

## 2 — Verify all eight generators
Checks the expected 1,500/1,500/500/500 files per generator and excludes zero-byte files. **All eight must pass before creating the final experiment manifests.**

In [ ]:
counts = []
for g in GENERATORS:
    print(f'\n{g}')
    for (split, label), expected in TARGETS.items():
        folder = DATA_ROOT / g / split / label
        images = ([p for p in folder.iterdir()
                   if p.is_file() and p.suffix.lower() in EXTS and p.stat().st_size > 0]
                  if folder.is_dir() else [])
        actual = len(images)
        counts.append({'generator':g, 'split':split, 'label':label,
                       'nonempty':actual, 'expected':expected})
        print(f'  {split}/{label}: {actual}/{expected} '
              + ('PASS' if actual == expected else 'INCOMPLETE'))

verification_df = pd.DataFrame(counts)
display(verification_df)
if not (verification_df.nonempty == verification_df.expected).all():
    raise RuntimeError('At least one folder is incomplete. Finish Midjourney or repair empty files first.')
print('\nPASS: All eight generators have 4,000 nonempty images (32,000 total).')

## 3 — Explore image counts and file types
Scans filenames, extensions, and nonzero file sizes. It does **not** decode every image.

In [ ]:
rows = []
for g in GENERATORS:
    for split in ['train', 'val']:
        for kind in ['ai', 'nature']:
            folder = DATA_ROOT / g / split / kind
            imgs = [p for p in folder.iterdir()
                    if p.is_file() and p.suffix.lower() in EXTS and p.stat().st_size > 0]
            rows.append({
                'generator': g, 'source_split': split,
                'class': kind, 'count': len(imgs),
                'extensions': dict(Counter(p.suffix.lower() for p in imgs))
            })
counts_df = pd.DataFrame(rows)
display(counts_df)
print('Total nonempty images:', counts_df['count'].sum())

## 4 — Create the balanced experiment manifests

Like the reference notebook, this produces **4 choices of k (1, 2, 4, 7) × 3 repeats = 12 experiments per held-out generator**, with **1,400 real + 1,400 AI train**, **280 + 280 validation**, and **280 + 280 held-out test** rows in each experiment.

- **Default:** hold out `ADM` to produce 12 experiments. Change `HOLDOUTS` to all 8 generators when ready to run the full 96-experiment design.
- The fake **training and validation** generators exclude the held-out generator. Fake **test** images come from the held-out generator only.
- Real images use a **single canonical `nature` pool** (`ADM` by default), avoiding variation in the real-image source across generator experiments. They are sampled without repeat across train/val/test **within** each experiment.
- Images are **not copied**. The output CSVs contain absolute paths to the original shared data.
- For each experiment, an SHA-256 audit checks that no exact file content appears across its train, val, and test splits. This reads image bytes from Drive and may take time or encounter Drive read quotas; if this happens, stop and retry after access recovers. **It does not detect near-duplicates.**

Results are written to your original shared compact dataset under `_experiment_manifests`. **Do not treat the final held-out test set as a hyperparameter-tuning set.**

In [ ]:
# Safe default: create 12 experiments holding out ADM only.
HOLDOUTS = ['ADM']
# To generate all 96 later, uncomment the next line:
# HOLDOUTS = GENERATORS.copy()

K_VALUES = [1, 2, 4, 7]
REPEATS = 3
CANONICAL_NATURE_GENERATOR = 'ADM'
SEED = 566

# Save CSVs to the owner's original shared folder; no images are copied.
OUT = DATA_ROOT / '_experiment_manifests'
OUT.mkdir(parents=True, exist_ok=True)
print('CSV output:', OUT)
print('Holdout generators:', HOLDOUTS)

### 4a — Manifest-building functions
Run this setup cell once. It creates no files until you run the following cell.

In [ ]:
def list_images(g, split, label):
    folder = DATA_ROOT / g / split / label
    if not folder.is_dir():
        raise FileNotFoundError(folder)
    images = sorted(p for p in folder.iterdir()
                    if p.is_file() and p.suffix.lower() in EXTS and p.stat().st_size > 0)
    return images

POOLS = {(g, s, lab): list_images(g,s,lab)
         for g in GENERATORS for s in ['train','val'] for lab in ['ai','nature']}

# SHA-256 cache avoids rereading files across experiments in this runtime.
HASH_CACHE = {}
def sha256(path):
    path = Path(path)
    key = str(path)
    if key not in HASH_CACHE:
        h = hashlib.sha256()
        with path.open('rb') as f:
            for chunk in iter(lambda: f.read(1024*1024), b''):
                h.update(chunk)
        HASH_CACHE[key] = h.hexdigest()
    return HASH_CACHE[key]


def balanced_samples(gens, split, count, rng):
    # Distribute AI examples evenly over the chosen generators.
    per_gen = [count // len(gens) + (i < count % len(gens)) for i in range(len(gens))]
    result = []
    for gen, n in zip(gens, per_gen):
        pool = POOLS[(gen,split,'ai')]
        if len(pool) < n:
            raise RuntimeError(f'Not enough AI images from {gen}/{split}: need {n}')
        result.extend((p, 1, gen, split) for p in rng.sample(pool, n))
    rng.shuffle(result)
    return result


def nature_samples(rng):
    # 2,000 real images total: 1,500 train + 500 val.
    # Allocate disjoint 1,400 train; 280 val; 280 test (1,960 overall).
    train_pool = POOLS[(CANONICAL_NATURE_GENERATOR, 'train', 'nature')]
    val_pool = POOLS[(CANONICAL_NATURE_GENERATOR, 'val', 'nature')]
    t = rng.sample(train_pool, 1400)
    v = rng.sample(val_pool, 280)
    remaining = [p for p in (train_pool + val_pool) if p not in set(t) and p not in set(v)]
    if len(remaining) < 280:
        raise RuntimeError('Canonical nature pool does not have enough distinct paths')
    test = rng.sample(remaining, 280)
    def items(arr):
        return [(p, 0, CANONICAL_NATURE_GENERATOR,
                 'train' if p in train_set else 'val') for p in arr]
    train_set = set(train_pool)
    return items(t), items(v), items(test)


def audit_and_frame(groups):
    # Inspect exact content for split leakage; additionally checks duplicates within a split.
    seen_hashes = {}
    frames = {}
    for split, items in groups.items():
        rows = []
        for path, label, gen, origin in items:
            digest = sha256(path)
            if digest in seen_hashes:
                other_split, other_path = seen_hashes[digest]
                raise RuntimeError(
                    'Exact-content overlap detected between ' + other_split + ' and ' + split
                    + ': ' + str(other_path) + ' / ' + str(path)
                )
            seen_hashes[digest] = (split, path)
            rows.append({'path':str(path), 'label':label,
                         'source_generator':gen, 'source_split':origin, 'sha256':digest})
        frames[split] = pd.DataFrame(rows)
    return frames

### 4b — Generate the CSVs and `experiments.json`
The first run can read many Drive files because it calculates checksums. If a Google Drive quota error appears, **do not delete anything**; keep the existing dataset unchanged and retry when access works. Partial experiment folders can be regenerated.

In [ ]:
experiments = []
for heldout in HOLDOUTS:
    assert heldout in GENERATORS
    candidates = [g for g in GENERATORS if g != heldout]

    for k in K_VALUES:
        for rep in range(REPEATS):
            experiment = f'holdout_{heldout}_k{k}_rep{rep}'
            rng = random.Random(f'{SEED}:{experiment}')
            train_gens = rng.sample(candidates, k)
            nat_train, nat_val, nat_test = nature_samples(rng)

            ai_train = balanced_samples(train_gens, 'train', 1400, rng)
            ai_val = balanced_samples(train_gens, 'val', 280, rng)
            ai_test = balanced_samples([heldout], 'val', 280, rng)

            groups = {
                'train': ai_train + nat_train,
                'val': ai_val + nat_val,
                'test': ai_test + nat_test,
            }
            for items in groups.values():
                rng.shuffle(items)

            print('Checking content hashes:', experiment, flush=True)
            frames = audit_and_frame(groups)

            dest = OUT / experiment
            dest.mkdir(parents=True, exist_ok=True)
            for split, frame in frames.items():
                frame.to_csv(dest / f'{split}.csv', index=False)

            experiments.append({
                'experiment':experiment, 'holdout':heldout,
                'k':k, 'rep':rep, 'train_generators':train_gens,
                'test_generator':heldout, 'nature_generator':CANONICAL_NATURE_GENERATOR,
                'counts':{s:len(df) for s,df in frames.items()}
            })
            (OUT / 'experiments.json').write_text(json.dumps(experiments, indent=2))

print(f'\nCreated {len(experiments)} experiments in {OUT}')

## 5 — Verify every generated split
Checks 1,400 real + 1,400 AI train, 280 + 280 validation, 280 + 280 test; checks fake-generator separation and the exact-hash column. This reuses the hashes stored during creation rather than rereading the images.

In [ ]:
assert (OUT / 'experiments.json').is_file(), 'Run section 4b first.'
exp_list = json.loads((OUT / 'experiments.json').read_text())
summary = []
for exp in exp_list:
    folder = OUT / exp['experiment']
    frames = {s: pd.read_csv(folder / f'{s}.csv') for s in ['train','val','test']}
    for split, df in frames.items():
        target = {0:1400,1:1400} if split == 'train' else {0:280,1:280}
        counts = df['label'].value_counts().to_dict()
        assert counts == target, f'Unbalanced {exp["experiment"]}/{split}: {counts}'
    train_gens = set(frames['train'].loc[frames['train'].label==1,'source_generator'])
    val_gens = set(frames['val'].loc[frames['val'].label==1,'source_generator'])
    test_gens = set(frames['test'].loc[frames['test'].label==1,'source_generator'])
    assert train_gens == val_gens == set(exp['train_generators'])
    assert test_gens == {exp['holdout']}
    assert test_gens.isdisjoint(train_gens)
    all_hashes = pd.concat([frames[s]['sha256'] for s in ['train','val','test']])
    assert all_hashes.is_unique, 'Exact-file overlap in an experiment'
    summary.append({'experiment':exp['experiment'], 'k':exp['k'],
                    'train_generators':', '.join(sorted(train_gens)),
                    'test_generator':exp['holdout'], 'status':'PASS'})

display(pd.DataFrame(summary))
print(f'PASS: {len(summary)} experiments verified.')

## 6 — Load a sample batch into PyTorch

This defines a minimal PyTorch dataset inside the notebook rather than requiring an external `dataset.py`. It converts images to RGB, resizes the shorter side to 256, center-crops to 224 × 224, and normalizes using ImageNet statistics. **No augmentation** is used for this baseline test.

Run the cell only after manifests exist. If PyTorch is not installed, install it in the runtime before rerunning.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class GenImageManifestDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        rec = self.df.iloc[idx]
        with Image.open(rec['path']) as im:
            image = self.transform(im.convert('RGB'))
        return image, int(rec['label'])

exp_name = exp_list[0]['experiment']
example_csv = OUT / exp_name / 'train.csv'
dataset = GenImageManifestDataset(example_csv)
loader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=0)
images, labels = next(iter(loader))
print('Experiment:', exp_name)
print('Images shape:', tuple(images.shape))
print('Labels:', labels.tolist())
assert tuple(images.shape[1:]) == (3,224,224)
print('PASS: PyTorch sample batch loaded.')

## 7 — Confirm the saved experiment manifests

The experiment CSVs are already saved in the **original shared** `GenImage_Compact/_experiment_manifests` directory. They are lightweight lists of image paths, not another copy of the 32,000 pictures. You can check the destination below.


In [ ]:
print('Shared dataset folder:', DATA_ROOT)
print('Saved manifests folder:', OUT)
print('Experiment folders:', len([p for p in OUT.iterdir() if p.is_dir()]))
print('Index:', OUT / 'experiments.json')

## 8 — Next checks before reporting results

- Inspect sampled image properties by class and generator (dimensions, formats, compression, distribution).
- Run a perceptual-hash and original source-ID **near-duplicate audit** across splits; exact SHA-256 hashes alone cannot detect resized or recompressed duplicates.
- For the complete benchmark, rerun Section 4 with **all eight holdouts** (96 experiment manifests total).
- For a stricter generator-family holdout, consider holding out both Stable Diffusion versions together.
- The real test images are sampled from the **canonical nature pool**, not necessarily from the held-out fake generator's real images.
- Keep held-out test results reserved for final reporting, not hyperparameter tuning.

**Important:** this notebook prepares and tests data-loading manifests. It does not train an image detector.